<a href="https://colab.research.google.com/github/cvip5559-afk/Fluvia/blob/main/AppleAiStutter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'Saved from Chrome (1)', 'Saved from Chrome', 'archive', 'Stutter_Detection-master']


In [3]:
archive_path = '/content/drive/MyDrive/archive'

print(os.listdir(archive_path))

['SEP-28k_labels.csv', 'SEP-28k_episodes.csv', 'fluencybank_episodes.csv', 'fluencybank_labels.csv', 'clips']


In [4]:
clips_path = '/content/drive/MyDrive/archive/clips'

print("عدد ملفات الصوت:", len(os.listdir(clips_path)))
print("أول 10 ملفات:", os.listdir(clips_path)[:10])

عدد ملفات الصوت: 1
أول 10 ملفات: ['stuttering-clips']


In [5]:
clips_path = "/content/drive/MyDrive/archive/clips/stuttering-clips/clips"

In [6]:
import os

print("عدد الملفات:", len(os.listdir(clips_path)))
print("أول 10 ملفات:", os.listdir(clips_path)[:10])

عدد الملفات: 32364
أول 10 ملفات: ['WomenWhoStutter_88_49.wav', 'WomenWhoStutter_88_5.wav', 'WomenWhoStutter_89_15.wav', 'WomenWhoStutter_91_26.wav', 'WomenWhoStutter_88_80.wav', 'WomenWhoStutter_88_98.wav', 'WomenWhoStutter_8_28.wav', 'WomenWhoStutter_90_33.wav', 'WomenWhoStutter_88_6.wav', 'WomenWhoStutter_90_7.wav']


In [7]:
from pathlib import Path
import csv
import random
import shutil
from collections import defaultdict

ROOT = Path("/content/drive/MyDrive/archive")
CLIPS_DIR = ROOT / "clips" / "stuttering-clips" / "clips"
OUTPUT_DIR = ROOT / "create_ml_dataset"

LABEL_FILES = [
    ROOT / "SEP-28k_labels.csv",
    ROOT / "fluencybank_labels.csv",
]

CLASSES = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection",
]

TRAIN_RATIO = 0.70
MIN_VOTES = 2
RANDOM_SEED = 42


def to_int(value):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return 0


def make_audio_name(show, episode_id, clip_id):
    show = str(show).strip()
    episode_text = str(episode_id).strip()
    clip_number = to_int(clip_id)

    if show.lower() == "fluencybank":
        # يحافظ على 010 كما هو، أو يحوله إلى 3 خانات
        episode_number = to_int(episode_text)
        return f"FluencyBank_{episode_number:03d}_{clip_number}.wav"

    episode_number = to_int(episode_text)
    return f"{show}_{episode_number}_{clip_number}.wav"


def choose_single_label(row):
    # استبعاد المقاطع غير المناسبة
    if to_int(row.get("PoorAudioQuality", 0)) >= 2:
        return None

    if to_int(row.get("NoSpeech", 0)) >= 2:
        return None

    if to_int(row.get("Unsure", 0)) >= 2:
        return None

    scores = {
        label: to_int(row.get(label, 0))
        for label in CLASSES
    }

    highest = max(scores.values(), default=0)

    winners = [
        label
        for label, score in scores.items()
        if score == highest and score > 0
    ]

    # نأخذ فقط تصنيفًا واضحًا حصل على صوتين أو ثلاثة، دون تعادل
    if highest >= MIN_VOTES and len(winners) == 1:
        return winners[0]

    return None


def load_grouped_files():
    grouped = defaultdict(list)
    skipped = 0
    missing = 0
    duplicate_keys = set()

    for labels_file in LABEL_FILES:
        print("Reading:", labels_file.name)

        with labels_file.open(
            "r",
            encoding="utf-8-sig",
            newline=""
        ) as file:
            reader = csv.DictReader(file)

            required = {"Show", "EpId", "ClipId"}
            columns = set(reader.fieldnames or [])

            if not required.issubset(columns):
                raise ValueError(
                    f"{labels_file.name} لا يحتوي الأعمدة المطلوبة. "
                    f"الأعمدة الموجودة: {reader.fieldnames}"
                )

            for row in reader:
                label = choose_single_label(row)

                if label is None:
                    skipped += 1
                    continue

                audio_name = make_audio_name(
                    row["Show"],
                    row["EpId"],
                    row["ClipId"]
                )

                audio_path = CLIPS_DIR / audio_name

                if not audio_path.exists():
                    missing += 1
                    continue

                key = (audio_name, label)

                if key in duplicate_keys:
                    continue

                duplicate_keys.add(key)
                grouped[label].append(audio_path)

    print("Skipped unclear files:", skipped)
    print("Missing audio files:", missing)

    return grouped


def prepare_output():
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    for split in ["train", "test"]:
        for label in CLASSES:
            (OUTPUT_DIR / split / label).mkdir(
                parents=True,
                exist_ok=True
            )


def split_and_copy(grouped):
    random.seed(RANDOM_SEED)

    for label in CLASSES:
        files = grouped.get(label, [])
        random.shuffle(files)

        split_index = int(len(files) * TRAIN_RATIO)
        train_files = files[:split_index]
        test_files = files[split_index:]

        for source in train_files:
            shutil.copy2(
                source,
                OUTPUT_DIR / "train" / label / source.name
            )

        for source in test_files:
            shutil.copy2(
                source,
                OUTPUT_DIR / "test" / label / source.name
            )

        print(
            f"{label}: total={len(files)}, "
            f"train={len(train_files)}, "
            f"test={len(test_files)}"
        )


prepare_output()
grouped = load_grouped_files()
split_and_copy(grouped)

print("\nتم الانتهاء ✅")
print("المجلد الناتج:")
print(OUTPUT_DIR)

Reading: SEP-28k_labels.csv
Reading: fluencybank_labels.csv
Skipped unclear files: 18139
Missing audio files: 0
Prolongation: total=2070, train=1449, test=621
Block: total=2427, train=1698, test=729
SoundRep: total=1873, train=1311, test=562
WordRep: total=2302, train=1611, test=691
Interjection: total=5510, train=3856, test=1654

تم الانتهاء ✅
المجلد الناتج:
/content/drive/MyDrive/archive/create_ml_dataset
